# Training con Time-Domain features 

Este NoteBook se centra en entrenar clasificadores con las características de dominio temporal de los datos, a diferencia de los otros, que entrenamos con datos crudos. 
El objetivo es comparar clasificadores con FE vs datos crudos, en aspectos como:
- Accuracy
- Latencia
- Tiempo de training
- Robustez al ruido

Aplicaremos cross validation para obtener unos promedios más confiables.

Además, compararemos algoritmos clásicos de ML con técnicas del estado del arte modernas.

In [1]:
import pandas as pd
import numpy as np
import time
from td_features import td_features_dataset
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score)

## Cargamos Datos

In [2]:
df = pd.read_csv("dataset.csv") # dataset completo
y = df.iloc[:, -1].values
df = df.drop(df.columns[-1], axis=1) # eliminamos label

## Pasamos a características del dominio temporal

In [3]:
df = td_features_dataset(df, scaler=StandardScaler())
X = df.values

## Splits

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Técnicas clásicas

### LDA

In [5]:
# LDA
n_classes = len(np.unique(y_train)) # para probar n_components
n_features = X_train.shape[1]
max_components = min(n_classes - 1, n_features)

param_grid = [
    {
        "solver" : ["lsqr", "eigen"],
        "shrinkage" : [None, "auto", 0.1, 0.5, 0.9],
        "n_components" : range(1, max_components + 1)
    }]

grid_lda = GridSearchCV(
    estimator=LinearDiscriminantAnalysis(),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

start = time.time()
grid_lda.fit(X_train, y_train)
end = time.time()
lda_training_time = end - start
best_lda = grid_lda.best_estimator_

print("Mejor estimador:", grid_lda.best_estimator_)
print("Mejores parámetros:", grid_lda.best_params_)
print("Mejor score CV:", grid_lda.best_score_)
print(f"Tiempo de training: {lda_training_time:.2f} s")


Mejor estimador: LinearDiscriminantAnalysis(n_components=1, shrinkage='auto', solver='lsqr')
Mejores parámetros: {'n_components': 1, 'shrinkage': 'auto', 'solver': 'lsqr'}
Mejor score CV: 0.901805426253491
Tiempo de training: 4.49 s


### KNN

In [6]:
# KNN

# KNN
param_grid = [
    {
        "n_neighbors": range(1, 30, 2),
        "weights": ["uniform", "distance"],
        "metric": ["euclidean", "manhattan", "minkowski"]
    }
]

grid_knn = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

start = time.time()
grid_knn.fit(X_train, y_train)
end = time.time()
knn_training_time = end - start
best_knn = grid_knn.best_estimator_

print("Mejor estimador:", grid_knn.best_estimator_)
print("Mejores parámetros:", grid_knn.best_params_)
print("Mejor score CV:", grid_knn.best_score_)
print(f"Tiempo de training: {knn_training_time:.2f} s")

Mejor estimador: KNeighborsClassifier(metric='manhattan', n_neighbors=19, weights='distance')
Mejores parámetros: {'metric': 'manhattan', 'n_neighbors': 19, 'weights': 'distance'}
Mejor score CV: 0.9292080324615549
Tiempo de training: 15.22 s


# Modelos ML clásicos

### SVM

In [7]:
param_grid = [
    {
        "kernel": ["rbf"],
        "C": [0.1, 1, 10, 100],
        "gamma": ["scale", "auto", 0.001, 0.01, 0.1],
    },
    {
        "kernel": ["poly"],
        "C": [0.1, 1, 10, 100],
        "gamma": ["scale", "auto"],
        "degree": [2, 3, 4],
        "coef0": [0.0, 1.0],
    },
]

grid = GridSearchCV(
    SVC(random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)

start = time.time()
grid.fit(X_train, y_train)
end = time.time()
svm_training_time = end - start
best_svm = grid.best_estimator_

print("Mejor estimador:", grid.best_estimator_)
print("Mejores parámetros:", grid.best_params_)
print("Mejor score CV:", grid.best_score_)
print(f"Tiempo de training: {svm_training_time:.2f} s")

Mejor estimador: SVC(C=100, gamma=0.001, random_state=42)
Mejores parámetros: {'C': 100, 'gamma': 0.001, 'kernel': 'rbf'}
Mejor score CV: 0.9663161168177872
Tiempo de training: 87.30 s


### Random Forest

In [8]:
param_grid = {
    'n_estimators':     [50, 100, 200],
    'max_features':     ['sqrt', 'log2', None],
    'max_depth':        [100, 10, 20],
    'min_samples_leaf': [1, 2, 5]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

start = time.time()
grid_rf.fit(X_train, y_train)
end = time.time()
rf_training_time = end - start
best_rf_model = grid_rf.best_estimator_

print(f"Mejores parámetros: {grid_rf.best_params_}")
print(f"Mejor CV score:     {grid_rf.best_score_:.4f}")
print(f"Test score:         {grid_rf.best_estimator_.score(X_test, y_test):.4f}")
print(f"Tiempo de training: {rf_training_time:.2f} s")

/home/jordi/Escritorio/uni/3/c2/Aprendizaje Avanzado/proyecto/entorno_proyecto/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/jordi/Escritorio/uni/3/c2/Aprendizaje Avanzado/proyecto/entorno_proyecto/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/jordi/Escritorio/uni/3/c2/Aprendizaje Avanzado/proyecto/entorno_proyecto/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagat

Mejores parámetros: {'max_depth': 100, 'max_features': 'log2', 'min_samples_leaf': 1, 'n_estimators': 200}
Mejor CV score:     0.9639
Test score:         0.9685
Tiempo de training: 247.91 s


### AdaBoost

In [9]:
param_grid = {
    'n_estimators':  [25, 50, 75, 100],
    'learning_rate': [0.01, 0.1, 0.5, 0.9],
    'estimator__max_depth' : [1, 2, 3, 5]
}

grid_ada = GridSearchCV(
    AdaBoostClassifier(estimator=DecisionTreeClassifier(), random_state=42),
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

start = time.time()
grid_ada.fit(X_train, y_train)
end = time.time()
ada_training_time = end - start
best_ada_model = grid_ada.best_estimator_

print(f"Mejores parámetros: {grid_ada.best_params_}")
print(f"Mejor CV score:     {grid_ada.best_score_:.4f}")
print(f"Test score:         {grid_ada.best_estimator_.score(X_test, y_test):.4f}")
print(f"Tiempo de training: {ada_training_time:.2f} s")

Mejores parámetros: {'estimator__max_depth': 5, 'learning_rate': 0.5, 'n_estimators': 100}
Mejor CV score:     0.9674
Test score:         0.9699
Tiempo de training: 115.78 s


### ANN

In [10]:
# red neuronal artificial (principio de deep learning básico)

param_grid = [
    {
        "hidden_layer_sizes": [(64,), (128,), (64, 64), (128, 64), (128, 128)],
        "activation": ["relu", "tanh"],
        "alpha": [0.0001, 0.001, 0.01],
        "learning_rate": ["constant", "adaptive"],
        "max_iter": [500]
    }
]

grid_ann = GridSearchCV(
    estimator=MLPClassifier(),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

start = time.time()
grid_ann.fit(X_train, y_train)
end = time.time()
ann_training_time = end - start
best_ann = grid_ann.best_estimator_

print("Mejor estimador:", grid_ann.best_estimator_)
print("Mejores parámetros:", grid_ann.best_params_)
print("Mejor score CV:", grid_ann.best_score_)
print(f"Tiempo de training: {ann_training_time:.2f} s")

Mejor estimador: MLPClassifier(activation='tanh', alpha=0.01, hidden_layer_sizes=(128,),
              learning_rate='adaptive', max_iter=500)
Mejores parámetros: {'activation': 'tanh', 'alpha': 0.01, 'hidden_layer_sizes': (128,), 'learning_rate': 'adaptive', 'max_iter': 500}
Mejor score CV: 0.9640327510176834
Tiempo de training: 400.94 s


## Ruido y Latencia

In [11]:
noise = np.random.uniform(-0.3*np.std(X_test), 0.3*np.std(X_test), X_test.shape)
X_noise = X_test + noise

In [12]:
def accuracy_and_latency(model, X_test, y_test):
    latencias = []
    for _ in range(5):
        start = time.time()
        y_pred = model.predict(X_test)
        end = time.time()
        latencias.append(end-start)
    elapsed = np.mean(latencias)
    acc = accuracy_score(y_test, y_pred)
    return [acc, elapsed]

In [13]:
# sin ruido
acc_svm, time_svm = accuracy_and_latency(best_svm, X_test, y_test)
acc_rf, time_rf = accuracy_and_latency(best_rf_model, X_test, y_test)
acc_ada, time_ada = accuracy_and_latency(best_ada_model, X_test, y_test)
acc_lda, time_lda = accuracy_and_latency(best_lda, X_test, y_test)
acc_knn, time_knn = accuracy_and_latency(best_knn, X_test, y_test)
acc_ann, time_ann = accuracy_and_latency(best_ann, X_test, y_test)

# con ruido
acc_svm_n, time_svm_n = accuracy_and_latency(best_svm, X_noise, y_test)
acc_rf_n, time_rf_n = accuracy_and_latency(best_rf_model, X_noise, y_test)
acc_ada_n, time_ada_n = accuracy_and_latency(best_ada_model, X_noise, y_test)
acc_lda_n, time_lda_n = accuracy_and_latency(best_lda, X_noise, y_test)
acc_knn_n, time_knn_n = accuracy_and_latency(best_knn, X_noise, y_test)
acc_ann_n, time_ann_n = accuracy_and_latency(best_ann, X_noise, y_test)

# latencias
lat_svm = (time_svm + time_svm_n) / 2
lat_rf = (time_rf + time_rf_n) / 2
lat_ada = (time_ada + time_ada_n) / 2
lat_lda = (time_lda + time_lda_n) / 2
lat_knn = (time_knn + time_knn_n) / 2
lat_ann = (time_ann + time_ann_n) / 2


pd.DataFrame({
    "Modelo": ["SVM", "Random Forest", "AdaBoost", "LDA", "KNN", "ANN"],
    "Accuracy": [f"{acc_svm:.4f}", f"{acc_rf:.4f}", f"{acc_ada:.4f}", f"{acc_lda:.4f}", f"{acc_knn:.4f}", f"{acc_ann:.4f}"],
    "Accuracy Ruido": [f"{acc_svm_n:.4f}", f"{acc_rf_n:.4f}", f"{acc_ada_n:.4f}", f"{acc_lda_n:.4f}", f"{acc_knn_n:.4f}", f"{acc_ann_n:.4f}"],
    "Latencia (s)": [f"{lat_svm:.4f}", f"{lat_rf:.4f}", f"{lat_ada:.4f}", f"{lat_lda:.4f}", f"{lat_knn:.4f}", f"{lat_ann:.4f}"],
    "Tiempo Training (s)": [f"{svm_training_time//60:.0f} min {svm_training_time%60:.0f}s",
                            f"{rf_training_time//60:.0f} min {rf_training_time%60:.0f}s",
                            f"{ada_training_time//60:.0f} min {ada_training_time%60:.0f}s",
                            f"{lda_training_time//60:.0f} min {lda_training_time%60:.0f}s",
                            f"{knn_training_time//60:.0f} min {knn_training_time%60:.0f}s",
                            f"{ann_training_time//60:.0f} min {ann_training_time%60:.0f}s"]
})

,Modelo,Accuracy,Accuracy Ruido,Latencia (s),Tiempo Training (s)
0,SVM,0.9702,0.9455,0.1283,1 min 27s
1,Random Forest,0.9685,0.9226,0.0330,4 min 8s
2,AdaBoost,0.9699,0.8918,0.0250,1 min 56s
3,LDA,0.8962,0.8702,0.0002,0 min 4s
4,KNN,0.9229,0.9188,0.1597,0 min 15s
5,ANN,0.9620,0.9329,0.0317,6 min 41s


# Estado del Arte